In [1]:
#===============================================================================
# Extract ancestry SNPs from AoU WGS using rsID -> GRCh38 mapping
# Output: RA_ancestry_SNPs.csv
#===============================================================================

import os
import time
import requests
import subprocess
import pandas as pd
import hail as hl

#-------------------------------------------------------------------------------
# 0) Setup
#-------------------------------------------------------------------------------
bucket = os.getenv("WORKSPACE_BUCKET")
mt_wgs_path = os.getenv("WGS_ACAF_THRESHOLD_SPLIT_HAIL_PATH")

hl.default_reference(new_default_reference="GRCh38")
mt = hl.read_matrix_table(mt_wgs_path)

print("Workspace bucket:", bucket)
print("WGS path:", mt_wgs_path)

Loading BokehJS ...

Initializing Hail with default parameters...
/opt/conda/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Running on Apache Spark version 3.5.3
SparkUI available at http://all-of-us-28470-m.us-central1-c.c.terra-vpc-sc-39ac9e8b.internal:39361
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /home/jupyter/workspaces/multimodeltestzhiyu/hail-20260425-1533-0.2.134-952ae203dbbe.log


Workspace bucket: gs://fc-secure-28df46b0-6f9d-4443-ae5f-cb0492e90c24
WGS path: gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/splitMT/hail.mt


2026-04-25 15:37:20.147 Hail: INFO: wrote matrix table with 152 rows and 245394 columns in 128 partitions to RA_ancestry_SNPs_filtered_by_rsid.mt
2026-04-25 15:37:24.687 Hail: INFO: Reading table to impute column types
2026-04-25 15:37:25.968 Hail: INFO: Finished type imputation
  Loading field 'snp_id' as type str (imputed)
  Loading field 'rsid' as type str (imputed)
2026-04-25 15:37:31.125 Hail: INFO: Coerced sorted dataset
2026-04-25 15:37:31.135 Hail: INFO: Coerced dataset with out-of-order partitions.
2026-04-25 15:37:31.663 Hail: INFO: Ordering unsorted dataset with network shuffle
2026-04-25 15:37:33.787 Hail: INFO: wrote table with 235 rows in 1 partition to /tmp/__iruid_314-uQoplArW4myiW3hUJGwKcf
2026-04-25 15:37:37.434 Hail: INFO: Coerced sorted dataset
2026-04-25 15:37:37.448 Hail: INFO: Coerced dataset with out-of-order partitions.
2026-04-25 15:37:47.639 Hail: INFO: Coerced sorted dataset
2026-04-25 15:37:47.644 Hail: INFO: Coerced dataset with out-of-order partitions.
20

In [3]:
#-------------------------------------------------------------------------------
# 1) Read ancestry SNP list from CSV instead of Excel
#-------------------------------------------------------------------------------

snp_csv = "Ancestry_SNP.csv"

# use skiprows=1 because the original Excel needed skiprows=1
aim_df = pd.read_csv(snp_csv, skiprows=1)

# clean column names
aim_df.columns = aim_df.columns.astype(str).str.strip()

aim_df = aim_df.rename(columns={
    "NCBI SNP Reference": "rsid",
    "Chr": "chr",
    "Location on NCBI Assembly": "pos_build36",
    "NCBI Assembly Build Number": "build",
    "NCBI Assembly Build Number ": "build",
    "VIC": "allele1",
    "FAM": "allele2"
})

print("Columns after rename:")
print(aim_df.columns.tolist())

aim_df = aim_df[["rsid", "chr", "pos_build36", "build", "allele1", "allele2"]]
aim_df = aim_df.dropna(subset=["rsid"])

aim_df["rsid"] = aim_df["rsid"].astype(str).str.strip()

target_rsids = (
    aim_df["rsid"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Number of target rsIDs:", len(target_rsids))
print("Build counts:")
print(aim_df["build"].value_counts(dropna=False))
print(target_rsids[:10])

Columns after rename:
['rsid', 'Assay ID', 'Strand', 'allele1', 'allele2', 'Context Sequence', 'chr', 'Celera ID', 'build', 'pos_build36', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', '1']
Number of target rsIDs: 128
Build counts:
build
36    128
Name: count, dtype: int64
['rs731257', 'rs2946788', 'rs3793451', 'rs10236187', 'rs1569175', 'rs2030763', 'rs8035124', 'rs4463276', 'rs316873', 'rs647325']


In [4]:
#-------------------------------------------------------------------------------
# 2) Save target rsIDs for documentation
#-------------------------------------------------------------------------------
rsid_df = pd.DataFrame({"rsid": target_rsids})
rsid_df.to_csv("RA_ancestry_target_rsids.tsv", sep="\t", index=False)

subprocess.run(
    ["gsutil", "cp", "RA_ancestry_target_rsids.tsv",
     f"{bucket}/data/RA_ancestry_target_rsids.tsv"],
    check=True
)

print("[INFO] Saved target rsID list.")

Copying file://RA_ancestry_target_rsids.tsv [Content-Type=text/tab-separated-values]...
/ [1 files][  1.2 KiB/  1.2 KiB]                                                
Operation completed over 1 objects/1.2 KiB.                                      


[INFO] Saved target rsID list.


In [5]:
#-------------------------------------------------------------------------------
# 3) Map rsIDs to GRCh38 coordinates/alleles using NCBI Variation API
#-------------------------------------------------------------------------------
import requests
import time

ncbi_chr_map = {
    "NC_000001.11": "chr1",  "NC_000002.12": "chr2",
    "NC_000003.12": "chr3",  "NC_000004.12": "chr4",
    "NC_000005.10": "chr5",  "NC_000006.12": "chr6",
    "NC_000007.14": "chr7",  "NC_000008.11": "chr8",
    "NC_000009.12": "chr9",  "NC_000010.11": "chr10",
    "NC_000011.10": "chr11", "NC_000012.12": "chr12",
    "NC_000013.11": "chr13", "NC_000014.9":  "chr14",
    "NC_000015.10": "chr15", "NC_000016.10": "chr16",
    "NC_000017.11": "chr17", "NC_000018.10": "chr18",
    "NC_000019.10": "chr19", "NC_000020.11": "chr20",
    "NC_000021.9":  "chr21", "NC_000022.11": "chr22",
    "NC_000023.11": "chrX",  "NC_000024.10": "chrY"
}

def map_one_rsid_to_grch38(rsid):
    rsnum = str(rsid).replace("rs", "").strip()
    url = f"https://api.ncbi.nlm.nih.gov/variation/v0/beta/refsnp/{rsnum}"
    
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return []
    
    js = r.json()
    out = []
    placements = js.get("primary_snapshot_data", {}).get("placements_with_allele", [])
    
    for pl in placements:
        traits = pl.get("placement_annot", {}).get("seq_id_traits_by_assembly", [])
        is_grch38 = any(
            ("GRCh38" in tr.get("assembly_name", "")) and tr.get("is_top_level", False)
            for tr in traits
        )
        if not is_grch38:
            continue
        
        for a in pl.get("alleles", []):
            spdi = a.get("allele", {}).get("spdi", {})
            seq_id = spdi.get("seq_id")
            zero_pos = spdi.get("position")
            ref = spdi.get("deleted_sequence")
            alt = spdi.get("inserted_sequence")
            
            if seq_id not in ncbi_chr_map:
                continue
            if ref is None or alt is None:
                continue
            if len(ref) != 1 or len(alt) != 1:
                continue
            if ref == alt:
                continue
            
            contig = ncbi_chr_map[seq_id]
            pos = int(zero_pos) + 1
            
            snp_id = (
                contig.replace("chr", "") + ":" +
                str(pos) + ":" + ref + ":" + alt
            )
            
            out.append({
                "rsid": "rs" + rsnum,
                "contig": contig,
                "pos": pos,
                "ref": ref,
                "alt": alt,
                "snp_id": snp_id
            })
    
    return out

mapped_rows = []

for i, rsid in enumerate(target_rsids, start=1):
    mapped_rows.extend(map_one_rsid_to_grch38(rsid))
    
    if i % 10 == 0:
        print(f"Mapped {i}/{len(target_rsids)} rsIDs...")
    
    time.sleep(0.1)

mapped_df = pd.DataFrame(mapped_rows).drop_duplicates()

print("Target rsIDs:", len(target_rsids))
print("Mapped GRCh38 SNP rows:", mapped_df.shape[0])
print("Unique mapped rsIDs:", mapped_df["rsid"].nunique())

mapped_df.to_csv("RA_ancestry_rsID_to_GRCh38_mapped.csv", index=False)

mapped_rsids = set(mapped_df["rsid"].unique())
unmatched_rsids = [x for x in target_rsids if x not in mapped_rsids]

print("Unmatched rsIDs:", len(unmatched_rsids))
print(unmatched_rsids[:20])

pd.DataFrame({"rsid": unmatched_rsids}).to_csv(
    "RA_ancestry_rsID_unmatched.csv",
    index=False
)

Mapped 10/128 rsIDs...
Mapped 20/128 rsIDs...
Mapped 30/128 rsIDs...
Mapped 40/128 rsIDs...
Mapped 50/128 rsIDs...
Mapped 60/128 rsIDs...
Mapped 70/128 rsIDs...
Mapped 80/128 rsIDs...
Mapped 90/128 rsIDs...
Mapped 100/128 rsIDs...
Mapped 110/128 rsIDs...
Mapped 120/128 rsIDs...
Target rsIDs: 128
Mapped GRCh38 SNP rows: 235
Unique mapped rsIDs: 128
Unmatched rsIDs: 0
[]


In [6]:
#===============================================================================
# 4) Filter AoU WGS by mapped GRCh38 intervals
#===============================================================================

mapped_df["interval"] = (
    mapped_df["contig"] + ":" +
    mapped_df["pos"].astype(str) + "-" +
    (mapped_df["pos"] + 1).astype(str)
)

target_intervals = mapped_df["interval"].drop_duplicates().tolist()

interval_exprs = [
    hl.parse_locus_interval(iv, reference_genome="GRCh38")
    for iv in target_intervals
]

mt_subset = hl.filter_intervals(mt, interval_exprs)

print("Rows after interval filtering:")
print(mt_subset.count())

Rows after interval filtering:


(164, 245394)


In [7]:
#===============================================================================
# 5) Exact allele matching using chr:pos:ref:alt
#===============================================================================

mt_subset = mt_subset.annotate_rows(
    snp_id =
        mt_subset.locus.contig.replace("chr", "") + ":" +
        hl.str(mt_subset.locus.position) + ":" +
        mt_subset.alleles[0] + ":" +
        mt_subset.alleles[1]
)

snp_ids = mapped_df["snp_id"].drop_duplicates().tolist()
snp_set = hl.set(snp_ids)

mt_filtered = mt_subset.filter_rows(
    snp_set.contains(mt_subset.snp_id)
)

mt_filtered = mt_filtered.checkpoint(
    "RA_ancestry_SNPs_filtered_by_rsid.mt",
    overwrite=True
)

print("Rows after exact SNP-ID filtering:")
print(mt_filtered.count())

Rows after exact SNP-ID filtering:
(152, 245394)


In [8]:
#===============================================================================
# 6) Add rsID annotation back to matched AoU WGS rows
#===============================================================================

mapped_key_df = mapped_df[["snp_id", "rsid"]].drop_duplicates()
mapped_key_df.to_csv("RA_ancestry_snpid_to_rsid.tsv", sep="\t", index=False)

subprocess.run(
    [
        "gsutil", "cp",
        "RA_ancestry_snpid_to_rsid.tsv",
        f"{bucket}/data/RA_ancestry_snpid_to_rsid.tsv"
    ],
    check=True
)

snpid_rsid_path = f"{bucket}/data/RA_ancestry_snpid_to_rsid.tsv"

snpid_rsid_ht = hl.import_table(
    snpid_rsid_path,
    delimiter="\t",
    impute=True
).key_by("snp_id")

mt_filtered = mt_filtered.annotate_rows(
    rsid = snpid_rsid_ht[mt_filtered.snp_id].rsid
)

mt_filtered.rows().key_by().select(
    "locus", "alleles", "snp_id", "rsid"
).show(20)

Copying file://RA_ancestry_snpid_to_rsid.tsv [Content-Type=text/tab-separated-values]...
/ [1 files][  5.9 KiB/  5.9 KiB]                                                
Operation completed over 1 objects/5.9 KiB.                                      


,,,
locus,alleles,snp_id,rsid
locus<GRCh38>,array<str>,str,str
chr1:6490316,"[""T"",""C""]","""1:6490316:T:C""","""rs2986742"""
chr1:12548144,"[""G"",""A""]","""1:12548144:G:A""","""rs6541030"""
chr1:12548144,"[""G"",""T""]","""1:12548144:G:T""","""rs6541030"""
chr1:17844391,"[""A"",""G""]","""1:17844391:A:G""","""rs647325"""
chr1:27605187,"[""G"",""A""]","""1:27605187:G:A""","""rs4908343"""
chr1:41894599,"[""G"",""A""]","""1:41894599:G:A""","""rs1325502"""
chr1:55197699,"[""A"",""G""]","""1:55197699:A:G""","""rs12130799"""
chr1:68384004,"[""A"",""C""]","""1:68384004:A:C""","""rs3118378"""


In [9]:
#===============================================================================
# 7) Prepare matched variant list for batched export
#===============================================================================

matched_rows = mt_filtered.rows().key_by().select(
    "locus", "alleles", "snp_id", "rsid"
).to_pandas()

matched_rows["snp_col"] = matched_rows["rsid"] + "_" + matched_rows["snp_id"]

matched_rows.to_csv("RA_ancestry_SNPs_matched_rows.csv", index=False)

print("Total matched AoU variant rows:", matched_rows.shape[0])
print("Unique rsIDs:", matched_rows["rsid"].nunique())
matched_rows.head()

Total matched AoU variant rows: 152
Unique rsIDs: 128


,locus,alleles,snp_id,rsid,snp_col
0,chr1:6490316,"[T, C]",1:6490316:T:C,rs2986742,rs2986742_1:6490316:T:C
1,chr1:12548144,"[G, A]",1:12548144:G:A,rs6541030,rs6541030_1:12548144:G:A
2,chr1:12548144,"[G, T]",1:12548144:G:T,rs6541030,rs6541030_1:12548144:G:T
3,chr1:17844391,"[A, G]",1:17844391:A:G,rs647325,rs647325_1:17844391:A:G
4,chr1:27605187,"[G, A]",1:27605187:G:A,rs4908343,rs4908343_1:27605187:G:A


In [10]:
#===============================================================================
# 8) Extract allele counts in batches of variant rows
#    Restart-safe version
#===============================================================================

import os
import gc
import subprocess

batch_size = 10   # safer than 20/30 for 245k participants
all_snp_ids = matched_rows["snp_id"].drop_duplicates().tolist()

n_batches = int((len(all_snp_ids) + batch_size - 1) / batch_size)

print("Total variant rows to export:", len(all_snp_ids))
print("Batch size:", batch_size)
print("Number of batches:", n_batches)

batch_files = []

for ibatch in range(n_batches):
    start = ibatch * batch_size
    end = min((ibatch + 1) * batch_size, len(all_snp_ids))

    batch_filename = f"RA_ancestry_SNPs_batch_{ibatch + 1:02d}.csv"
    bucket_path = f"{bucket}/data/{batch_filename}"

    # Skip if already saved locally
    if os.path.exists(batch_filename):
        print(f"[Batch {ibatch + 1}/{n_batches}] local file exists, skipping: {batch_filename}")
        batch_files.append(batch_filename)
        continue

    # Skip if already saved in bucket
    check_bucket = subprocess.run(
        ["gsutil", "ls", bucket_path],
        capture_output=True,
        text=True
    )
    if check_bucket.returncode == 0:
        print(f"[Batch {ibatch + 1}/{n_batches}] bucket file exists, skipping: {bucket_path}")
        batch_files.append(batch_filename)
        continue

    batch_snp_ids = all_snp_ids[start:end]
    batch_set = hl.literal(set(batch_snp_ids))

    print(f"\n[Batch {ibatch + 1}/{n_batches}] exporting variants {start + 1}-{end}")

    mt_batch = mt_filtered.filter_rows(
        batch_set.contains(mt_filtered.snp_id)
    )

    mt_batch = mt_batch.annotate_entries(
        allele_count = hl.case()
            .when(mt_batch.GT.is_hom_ref(), 0)
            .when(mt_batch.GT.is_het(), 1)
            .when(mt_batch.GT.is_hom_var(), 2)
            .or_missing()
    )

    mt_batch = mt_batch.annotate_rows(
        snp_col = mt_batch.rsid + "_" + mt_batch.snp_id
    )

    table = mt_batch.entries()
    table = table.key_by()
    table = table.select("s", "snp_col", "allele_count")

    snp_matrix_batch = table.to_pandas().pivot(
        index="s",
        columns="snp_col",
        values="allele_count"
    )

    snp_matrix_batch = snp_matrix_batch.reset_index().rename(columns={"s": "person_id"})
    snp_matrix_batch["person_id"] = snp_matrix_batch["person_id"].astype(int)

    snp_matrix_batch.to_csv(batch_filename, index=False)
    batch_files.append(batch_filename)

    # Copy each batch file to bucket immediately
    output = subprocess.run(
        ["gsutil", "cp", f"./{batch_filename}", bucket_path],
        capture_output=True,
        text=True
    )

    print(output.stderr)
    print("Saved:", batch_filename, "shape:", snp_matrix_batch.shape)

    # Clean Python-side memory before next batch
    del mt_batch, table, snp_matrix_batch
    gc.collect()

print("\nFinished batch export.")
print("Batch files:")
print(batch_files)

Total variant rows to export: 152
Batch size: 10
Number of batches: 16

[Batch 1/16] exporting variants 1-10


Copying file://./RA_ancestry_SNPs_batch_01.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_01.csv shape: (245394, 11)

[Batch 2/16] exporting variants 11-20


Copying file://./RA_ancestry_SNPs_batch_02.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_02.csv shape: (245394, 11)

[Batch 3/16] exporting variants 21-30


Copying file://./RA_ancestry_SNPs_batch_03.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_03.csv shape: (245394, 11)

[Batch 4/16] exporting variants 31-40


Copying file://./RA_ancestry_SNPs_batch_04.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_04.csv shape: (245394, 11)

[Batch 5/16] exporting variants 41-50


...

Copying file://./RA_ancestry_SNPs_batch_12.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_12.csv shape: (245394, 11)

[Batch 13/16] exporting variants 121-130


Copying file://./RA_ancestry_SNPs_batch_13.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_13.csv shape: (245394, 11)

[Batch 14/16] exporting variants 131-140


Copying file://./RA_ancestry_SNPs_batch_14.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_14.csv shape: (245394, 11)

[Batch 15/16] exporting variants 141-150


Copying file://./RA_ancestry_SNPs_batch_15.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  6.6 MiB]                                                
/ [1 files][  6.6 MiB/  6.6 MiB]                                                
Operation completed over 1 objects/6.6 MiB.                                      

Saved: RA_ancestry_SNPs_batch_15.csv shape: (245394, 11)

[Batch 16/16] exporting variants 151-152


Copying file://./RA_ancestry_SNPs_batch_16.csv [Content-Type=text/csv]...
/ [0 files][    0.0 B/  2.8 MiB]                                                
/ [1 files][  2.8 MiB/  2.8 MiB]                                                
Operation completed over 1 objects/2.8 MiB.                                      

Saved: RA_ancestry_SNPs_batch_16.csv shape: (245388, 3)

Finished batch export.
Batch files:
['RA_ancestry_SNPs_batch_01.csv', 'RA_ancestry_SNPs_batch_02.csv', 'RA_ancestry_SNPs_batch_03.csv', 'RA_ancestry_SNPs_batch_04.csv', 'RA_ancestry_SNPs_batch_05.csv', 'RA_ancestry_SNPs_batch_06.csv', 'RA_ancestry_SNPs_batch_07.csv', 'RA_ancestry_SNPs_batch_08.csv', 'RA_ancestry_SNPs_batch_09.csv', 'RA_ancestry_SNPs_batch_10.csv', 'RA_ancestry_SNPs_batch_11.csv', 'RA_ancestry_SNPs_batch_12.csv', 'RA_ancestry_SNPs_batch_13.csv', 'RA_ancestry_SNPs_batch_14.csv', 'RA_ancestry_SNPs_batch_15.csv', 'RA_ancestry_SNPs_batch_16.csv']


In [11]:
#===============================================================================
# 9) Save matched rows and mapping summaries
#===============================================================================

for filename in [
    "RA_ancestry_SNPs_matched_rows.csv",
    "RA_ancestry_rsID_to_GRCh38_mapped.csv",
    "RA_ancestry_rsID_unmatched.csv",
    "RA_ancestry_target_rsids.tsv",
    "RA_ancestry_snpid_to_rsid.tsv"
]:
    args = ["gsutil", "cp", f"./{filename}", f"{bucket}/data/"]
    output = subprocess.run(args, capture_output=True)
    print(output.stderr.decode())

print("[INFO] Saved all ancestry SNP batch files and summaries.")
print("Batch files:")
print(batch_files)

Copying file://./RA_ancestry_SNPs_matched_rows.csv [Content-Type=text/csv]...
/ [1 files][ 11.7 KiB/ 11.7 KiB]                                                
Operation completed over 1 objects/11.7 KiB.                                     

Copying file://./RA_ancestry_rsID_to_GRCh38_mapped.csv [Content-Type=text/csv]...
/ [1 files][ 10.2 KiB/ 10.2 KiB]                                                
Operation completed over 1 objects/10.2 KiB.                                     

Copying file://./RA_ancestry_rsID_unmatched.csv [Content-Type=text/csv]...
/ [1 files][    5.0 B/    5.0 B]                                                
Operation completed over 1 objects/5.0 B.                                        

Copying file://./RA_ancestry_target_rsids.tsv [Content-Type=text/tab-separated-values]...
/ [1 files][  1.2 KiB/  1.2 KiB]                                                
Operation completed over 1 objects/1.2 KiB.                                      

Copying file://./RA